In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [2]:
data_path = "../../data/processed/fraud_merged_raw.pkl"

fraud_df = pd.read_pickle(data_path)

print("Dataset loaded successfully.")
print("Shape:", fraud_df.shape)

Dataset loaded successfully.
Shape: (590540, 436)


In [3]:
# Remove features with more than 90% missing values

missing_percentage = fraud_df.isnull().mean() * 100

high_missing_cols = missing_percentage[
    missing_percentage > 90
].index.tolist()

print(f"Columns with >90% missing values: {len(high_missing_cols)}")

fraud_df = fraud_df.drop(columns=high_missing_cols)

print("New shape:", fraud_df.shape)

Columns with >90% missing values: 12
New shape: (590540, 424)


In [4]:
identifier_columns = [
    "TransactionID",
    "TransactionDT"
]

fraud_df = fraud_df.drop(columns=identifier_columns)

print(f"Shape after removing identifiers: {fraud_df.shape}")

Shape after removing identifiers: (590540, 422)


In [5]:
X = fraud_df.drop(columns=["isFraud"])
y = fraud_df["isFraud"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (590540, 421)
Target: (590540,)


In [6]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Numerical features :", len(numerical_features))
print("Categorical features :", len(categorical_features))

Numerical features : 392
Categorical features : 29


In [7]:
# Train-test split before imputation/encoding to avoid data leakage

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (472432, 421)
X_test : (118108, 421)
y_train: (472432,)
y_test : (118108,)


In [8]:
# Confirm fraud distribution is preserved after stratified split

print("Training target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training target distribution:
isFraud
0    96.501084
1     3.498916
Name: proportion, dtype: float64

Testing target distribution:
isFraud
0    96.50066
1     3.49934
Name: proportion, dtype: float64


In [9]:
# Numerical imputation using training medians only

numerical_medians = X_train[numerical_features].median()

X_train[numerical_features] = X_train[numerical_features].fillna(numerical_medians)
X_test[numerical_features] = X_test[numerical_features].fillna(numerical_medians)

print("Missing numerical values in train:", X_train[numerical_features].isnull().sum().sum())
print("Missing numerical values in test :", X_test[numerical_features].isnull().sum().sum())

Missing numerical values in train: 0
Missing numerical values in test : 0


In [10]:
# Categorical imputation using "Unknown"

X_train[categorical_features] = X_train[categorical_features].fillna("Unknown")
X_test[categorical_features] = X_test[categorical_features].fillna("Unknown")

print("Missing categorical values in train:", X_train[categorical_features].isnull().sum().sum())
print("Missing categorical values in test :", X_test[categorical_features].isnull().sum().sum())

Missing categorical values in train: 0
Missing categorical values in test : 0


In [11]:
# Label encoding categorical variables

label_encoders = {}

for col in categorical_features:
    encoder = LabelEncoder()
    
    combined_values = pd.concat([
        X_train[col].astype(str),
        X_test[col].astype(str)
    ], axis=0)
    
    encoder.fit(combined_values)
    
    X_train[col] = encoder.transform(X_train[col].astype(str))
    X_test[col] = encoder.transform(X_test[col].astype(str))
    
    label_encoders[col] = encoder

print("Categorical encoding complete.")
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

Categorical encoding complete.
X_train shape: (472432, 421)
X_test shape : (118108, 421)


In [12]:
print("Remaining object columns in train:", X_train.select_dtypes(include=["object", "string"]).shape[1])
print("Remaining object columns in test :", X_test.select_dtypes(include=["object", "string"]).shape[1])

Remaining object columns in train: 0
Remaining object columns in test : 0


In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Scaling completed.")

Scaling completed.


In [14]:
print("Scaled training shape :", X_train_scaled.shape)
print("Scaled testing shape  :", X_test_scaled.shape)

Scaled training shape : (472432, 421)
Scaled testing shape  : (118108, 421)


In [15]:
import os
import joblib

os.makedirs("../../data/processed", exist_ok=True)

X_train.to_pickle("../../data/processed/X_train.pkl")
X_test.to_pickle("../../data/processed/X_test.pkl")

X_train_scaled.to_pickle("../../data/processed/X_train_scaled.pkl")
X_test_scaled.to_pickle("../../data/processed/X_test_scaled.pkl")

y_train.to_pickle("../../data/processed/y_train.pkl")
y_test.to_pickle("../../data/processed/y_test.pkl")

joblib.dump(label_encoders, "../../data/processed/label_encoders.pkl")
joblib.dump(scaler, "../../data/processed/standard_scaler.pkl")

print("All preprocessing artifacts saved successfully.")

All preprocessing artifacts saved successfully.
